<a href="https://colab.research.google.com/github/arnabd64/Google-Colab-Notebooks/blob/main/Pretrain_GPT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install --upgrade transformers[torch] datasets tensorboard pyarrow

from IPython.display import clear_output
clear_output()

In [ ]:
from google.colab import userdata
import huggingface_hub

huggingface_hub.login(token=userdata.get("HF"))

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [ ]:
from dataclasses import dataclass

class Config:
    # dataset params
    dataset_id: str = "HuggingFaceFW/fineweb"
    dataset_config: str = "default"
    dataset_split: str = "train"

    # model params
    model_id: str = "gpt2"
    context_length: int = 2048

In [ ]:
import datasets
fineweb = datasets.load_dataset(Config.dataset_id, Config.dataset_config, split=Config.dataset_split, streaming=True)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/23781 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/23781 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(Config.model_id)
tokenizer.pad_token = tokenizer.eos_token


def tokenize(batch):
    # extract the text
    # text = [sample['text'] for sample in batch]

    # tokenize the batch
    tokenized_batch = tokenizer(
        batch['text'],
        truncation = True,
        padding = "max_length",
        max_length = Config.context_length,
        return_tensors = "pt"
    )

    # return only the necessary keys
    return {
        "input_ids": tokenized_batch["input_ids"].squeeze(),
        "attention_mask": tokenized_batch["attention_mask"].squeeze()
    }


# apply the tokenization to the dataset
fineweb = fineweb.map(tokenize)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    # directories
    output_dir = "./gpt2-fineweb",
    overwrite_output_dir = True,
    logging_dir = "logs",

    # batch and training length
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,

    # learning rate
    learning_rate = 1.5e-5,
    lr_scheduler_type = "constant",
    warmup_steps = 0,

    # logging, eval and saving
    max_steps = 25000,
    logging_strategy = "steps",
    logging_steps = 250,
    eval_strategy = "no",
    save_strategy = "steps",
    save_steps = 5000,
    save_total_limit = 3,
    push_to_hub = True,

    # others
    report_to = ["tensorboard"],
    skip_memory_metrics = False,
    seed = 1337
)

In [ ]:
from transformers import AutoModelForCausalLM, Trainer, DataCollatorForLanguageModeling

model = AutoModelForCausalLM.from_pretrained(Config.model_id)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, return_tensors="pt")

trainer = Trainer(
    args = args,
    model = model,
    tokenizer = tokenizer,
    data_collator = data_collator,
    train_dataset = fineweb
)

trainer.train()

In [ ]:
# ! export CUDA_LAUNCH_BLOCKING=1
trainer.train()